<a href="https://colab.research.google.com/github/TOTVScontext/DataScience/blob/main/Context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!python -m spacy download pt_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/pt_core_news_sm-3.8.0/pt_core_news_sm-3.8.0-py3-none-any.whl (13.0 MB)
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [22]:
import pandas as pd
import re
import nltk
from collections import Counter
import plotly.express as px
import spacy
from IPython.display import display

nlp = spacy.load("pt_core_news_sm")
nltk.download('stopwords')
stop_words_pt = set(nltk.corpus.stopwords.words('portuguese'))

stop_words_custom = {
    'pra', 'aí', 'então', 'né', 'tá', 'gente', 'aqui', 'vai', 'fazer',
    'ter', 'porque', 'lá', 'assim', 'ser', 'vou', 'agora', 'tudo',
    'bem', 'entendi', 'falou', 'acho', 'vamos', 'pode', 'sobre', 'dá',
    'voc', 'você', 'n', 't', 'pro', 'mim', 'coisa', 'parte', 'dia', 'exemplo',
    'cara'
}
stop_words_pt.update(stop_words_custom)

df = pd.read_csv('ANON_nome_transcricao.csv', sep=',', on_bad_lines='skip', encoding='utf-8', encoding_errors='ignore')


#LIMPEZA E EXTRAÇÃO

def limpar_transcricao(texto):
    if pd.isna(texto):
        return ""
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúãõçâêôà\s]', ' ', texto)
    texto = ' '.join(texto.split())
    return texto

def extrair_palavras_chave(texto):
    palavras = texto.split()
    palavras_limpas = [p for p in palavras if p not in stop_words_pt and len(p) > 2]
    return palavras_limpas

def extrair_palavras_chave_spacy(texto):
    doc = nlp(texto)
    palavras_limpas = [
        token.text for token in doc
        if token.text not in stop_words_pt
        and len(token.text) > 2
        and token.pos_ == "NOUN"
    ]

    return palavras_limpas

#Texto limpo
print("Aplicando limpeza pesada e processando vocabulário...")
df['TEXTO_LIMPO'] = df['ANON_TRANSCRICAO'].apply(limpar_transcricao)
#logica de gatilhos (proximos possiveis passos)
gatilhos = 'whatsapp|ligar|contato|agendar|retorno|proxima|numero|reuniao'
df['TEM_PROXIMO_PASSO'] = df['TEXTO_LIMPO'].str.contains(gatilhos, case=False, na=False)

print("--- CONTADOR DE GATILHOS ---")
print(df['TEM_PROXIMO_PASSO'].value_counts())
print("\n--- AMOSTRA DOS 10 PRIMEIROS GATILHOS ENCONTRADOS ---")
display(df[df['TEM_PROXIMO_PASSO'] == True][['TEXTO_LIMPO']].head(10))

#TExto limpo so de substantivos essenciais
df['TOKENS'] = df['TEXTO_LIMPO'].apply(extrair_palavras_chave_spacy)

#TERMOS MAIS FREQUENTES

todas_palavras = [palavra for sublist in df['TOKENS'] for palavra in sublist]
contagem = Counter(todas_palavras)

top_palavras = pd.DataFrame(contagem.most_common(20), columns=['Palavra', 'Frequência'])

fig_palavras = px.bar(top_palavras, x='Frequência', y='Palavra',
                      orientation='h',
                      title='Top 20 Termos Reais de Negócios (Dados Limpos)',
                      color='Frequência',
                      color_continuous_scale=px.colors.sequential.Teal)
fig_palavras.update_layout(yaxis={'categoryorder':'total ascending'})
fig_palavras.show()








[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Aplicando limpeza pesada e processando vocabulário...
--- CONTADOR DE GATILHOS ---
TEM_PROXIMO_PASSO
False    327515
True         87
Name: count, dtype: int64

--- AMOSTRA DOS 10 PRIMEIROS GATILHOS ENCONTRADOS ---


,TEXTO_LIMPO
6898,grupos de whatsapp
8370,nessa questão que a gente está esperando o ret...
12610,o consultor continua atendendo ele via whatsapp
13642,pode ser feito tudo pelo whatsapp
14358,ok terminamos isso qual é a próxima que nós va...
16928,depois conversando vocês podem me ligar
18963,vocês tiveram contato com o nosso
22610,entendeu e eu quero ter essa aproximação eu se...
24472,eu posso ligar para a e falar
24787,a gente conseguia absorver isso no contato que...
